In [1]:
# Imports
import pandas as pd
import geopandas as gpd
import folium
import h3

In [2]:
# Basic file extension extraction
filename = "example_file.csv"
ext = filename.lower().split(".")[-1]
print(ext)

csv


In [3]:
# Load your NYC boundary as a GeoJSON polygon
nyc_geojson = {
    "type": "Polygon",
    "coordinates": [[[-74.15037856326225, 40.83225535896091], 
                     [-73.795026216383, 40.83225535896091], 
                     [-73.795026216383, 40.66873583662044], 
                     [-74.15037856326225, 40.66873583662044], 
                     [-74.15037856326225, 40.83225535896091]]]
}
# GeoJSON coords are (lng, lat) - need to flip to (lat, lng) for H3
coords = nyc_geojson["coordinates"][0]
latlng_coords = [(lat, lng) for lng, lat in coords]

nyc_poly = h3.LatLngPoly(latlng_coords)

# Get all H3 cell indexes at resolution 8
nyc_hexes = h3.polygon_to_cells(nyc_poly, res=8)
print(f"Generated {len(nyc_hexes)} cells over NYC.")

Generated 733 cells over NYC.


In [4]:
# Convert H3 list to GeoDataFrame
nyc_gdf = gpd.GeoDataFrame({'h3_index': list(nyc_hexes)})

# Function to return H3 boundar 
def cell_to_boundary(h3_index):
    boundary = h3.cell_to_boundary(h3_index)
    return boundary

# Display results
nyc_gdf['boundaries'] = nyc_gdf['h3_index'].apply(cell_to_boundary)
nyc_gdf.head(2)

,h3_index,boundaries
0,882a10722bfffff,"((40.76049785292696, -74.05036588048738), (40...."
1,882a100f29fffff,"((40.790498402455505, -73.91877507487021), (40..."


In [5]:
# Map
m = folium.Map(location=[40.7128, -74.0060], zoom_start=10, tiles='cartodbpositron')

# Add NYC boundary to the map
for _, row in nyc_gdf.iterrows():
    boundary = row['boundaries']
    folium.Polygon(locations=boundary, 
                   color='grey', 
                   fill=True, 
                   fill_opacity=0.25
                   ).add_to(m)

m